In [14]:
from IPython.core import display_functions
from IPython.core import display_functions
from sentence_transformers.sparse_encoder.losses import SparseMultipleNegativesRankingLoss
import pandas as pd
import requests
from ragas import EvaluationDataset
from ragas import evaluate
from ragas.llms import LangchainLLMWrapper
from ragas.metrics.collections import faithfulness, answer_correctness
from langchain_groq import ChatGroq
from dotenv import load_dotenv
import os
from ragas.run_config import RunConfig
from ragas.llms import llm_factory
from ragas.metrics import LLMContextRecall, Faithfulness
from langchain_google_genai import ChatGoogleGenerativeAI
from datasets import Dataset 
from langchain_community.embeddings import SentenceTransformerEmbeddings
from google import genai

import time

/var/folders/rg/bf6jgyyx59q1398512l2gxv40000gn/T/ipykernel_1741/3564749668.py:15: DeprecationWarning: Importing LLMContextRecall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import LLMContextRecall
  from ragas.metrics import LLMContextRecall, Faithfulness
/var/folders/rg/bf6jgyyx59q1398512l2gxv40000gn/T/ipykernel_1741/3564749668.py:15: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Faithfulness
  from ragas.metrics import LLMContextRecall, Faithfulness


In [7]:
load_dotenv()

df = pd.read_csv("testset.csv")
user_queries = df["user_input"]
expected_responses = df["reference"]

dataset = []

for query,reference in zip(user_queries,expected_responses):
    print(f'Processing query: {query}')
    time.sleep(5)  # Pause to avoid rate limits

    response = requests.post("http://localhost:8000/evaluation_query", json={"prompt": query, "history": []})
    response_json = response.json()

    relevant_docs = [doc["page_content"] for doc in response_json["source_chunks"]]
    response = response_json["answer"]
    dataset.append(
        {
            "user_input":query,
            "retrieved_contexts":relevant_docs,
            "response":response,
            "reference":reference
        }
    )

Processing query: What is the significance of Makerere Hill in Kampala's geography and history?
Processing query: Uganda's National Parks wildlife essay
Processing query: What are the major cultural and architectural highlights in Kampala, including the Uganda National Mosque (Gaddafi Mosque) and the Uganda Museum, and how do they illustrate the city's religious diversity and heritage?
Processing query: Could you descrbe how the Buganda Kngdom's cultural heritage shows up in Kampala, especially regarding the Lubiri Palac of the Kabaka and the nearby Kasubi Tombs that serve as burial grounds for the kingdom's kings?
Processing query: What kind of evening entertainment Ndere Cultural Centre give and how it show Ugandan dance and music?
Processing query: As a cultural heritage tour guide, how would you describe the role of dancehall music in Kampala's nightlife and its significance for visitors seeking to experience modern Ugandan culture?
Processing query: How does Water Hyacint affect L

In [12]:
evaluation_dataset = EvaluationDataset.from_list(dataset)

In [8]:
print(dataset)

[{'user_input': "What is the significance of Makerere Hill in Kampala's geography and history?", 'retrieved_contexts': ['Discovering Kampala: A City of Hills, Culture, and Energy\n\nRising across a series of rolling hills near the northern shores of Lake Victoria, Kampala is the vibrant capital of Uganda. It is a city where tradition and modern life blend seamlessly—where boda-bodas weave through traffic, markets buzz with energy, and religious landmarks stand as symbols of deep-rooted history.\n\nA City Built on Seven Hills\n\nKampala was originally built on seven hills, though it has since expanded far beyond them. Among the most notable are Kololo Hill, Makerere Hill, and Namirembe Hill. These hills not only shape the city’s geography but also reflect its colonial and cultural history.\n\nCultural and Historical Significance', 'Neighborhoods like Kololo and Kabalagala are hotspots for restaurants, bars, and clubs, reflecting a youthful and energetic population.\n\nChallenges and Gro

[{'user_input': "What is the significance of Makerere Hill in Kampala's geography and history?", 'retrieved_contexts': ['Discovering Kampala: A City of Hills, Culture, and Energy\n\nRising across a series of rolling hills near the northern shores of Lake Victoria, Kampala is the vibrant capital of Uganda. It is a city where tradition and modern life blend seamlessly—where boda-bodas weave through traffic, markets buzz with energy, and religious landmarks stand as symbols of deep-rooted history.\n\nA City Built on Seven Hills\n\nKampala was originally built on seven hills, though it has since expanded far beyond them. Among the most notable are Kololo Hill, Makerere Hill, and Namirembe Hill. These hills not only shape the city’s geography but also reflect its colonial and cultural history.\n\nCultural and Historical Significance', 'Neighborhoods like Kololo and Kabalagala are hotspots for restaurants, bars, and clubs, reflecting a youthful and energetic population.\n\nChallenges and Gro

In [16]:
evaluator_llm = LangchainLLMWrapper(ChatGroq(
    model="openai/gpt-oss-120b",
    api_key=os.getenv("GROQ_EVAL_KEY"),
    temperature=0
))

/var/folders/rg/bf6jgyyx59q1398512l2gxv40000gn/T/ipykernel_1741/3044729880.py:1: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  evaluator_llm = LangchainLLMWrapper(ChatGroq(


In [17]:


result = evaluate(dataset=evaluation_dataset,metrics=[LLMContextRecall(), Faithfulness()],llm=evaluator_llm,run_config=RunConfig(
        max_workers=1,   # 👈 one request at a time
        timeout=180,
        max_retries=10
    ))

Evaluating: 100%|██████████| 42/42 [21:22<00:00, 30.54s/it]


In [18]:
print(result)

{'context_recall': 0.9628, 'faithfulness': 0.4033}
